# 01 Data Exploration & Loading

Load S&P 500 data, compute log returns, and explore statistical properties.

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.volatility_forecasting.data.loader import download_data
from src.volatility_forecasting.data.preprocessor import DataPreprocessor
from src.volatility_forecasting.logger import setup_logger

logger = setup_logger('notebook')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Download S&P 500 data (2010-2024)
price_data = download_data(
    ticker='^GSPC',
    start='2010-01-01',
    end='2024-12-31'
)
print(f"Downloaded {len(price_data)} price observations")
print(f"Date range: {price_data.index.min()} to {price_data.index.max()}")
price_data.head()

In [ ]:
# Compute log returns
preprocessor = DataPreprocessor(price_data)
returns = preprocessor.compute_log_returns()
returns = preprocessor.remove_na()

print(f"Total returns: {len(returns)}")
print(f"Mean return: {returns.mean():.6f}%")
print(f"Std dev: {returns.std():.6f}%")
print(f"Skewness: {returns.skew():.4f}")
print(f"Kurtosis: {returns.kurtosis():.4f}")

In [ ]:
# Visualize returns
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Time series
axes[0, 0].plot(returns.index, returns.values)
axes[0, 0].set_title('Log Returns Over Time')
axes[0, 0].set_ylabel('Return (%)')

# Distribution
axes[0, 1].hist(returns, bins=50, edgecolor='black')
axes[0, 1].set_title('Distribution of Returns')
axes[0, 1].set_xlabel('Return (%)')

# ACF
axes[1, 0].bar(range(1, 21), [returns.autocorr(lag=i) for i in range(1, 21)])
axes[1, 0].set_title('Autocorrelation')
axes[1, 0].set_xlabel('Lag')

# Squared returns (volatility clustering)
axes[1, 1].plot(returns.index, returns**2)
axes[1, 1].set_title('Squared Returns (Volatility Clustering)')
axes[1, 1].set_ylabel('Return² (%²)')

plt.tight_layout()
plt.savefig('../report/figures/01_eda.png', dpi=150, bbox_inches='tight')
plt.show()

logger.info(f"EDA complete. {len(returns)} observations ready for modeling.")